# dGbyG--ThermoInfer workflow for Yeast-GEM

This notebook contains the JupyterLab-based steps of the workflow. The complete workflow consists of four main stages:

1. Configure the shared computational environment for dGbyG and ThermoInfer.
2. Use this notebook to load the GEM and define compartment-specific conditions.
3. Use this notebook to run dGbyG and save the reaction-level transformed standard Gibbs energy table as `Yeast9_standard_dGr_dGbyG.csv`.
4. Perform TFBA and identify candidate reactions with inconsistent directionality.

    4.1. Pause this notebook and run the standalone ThermoInfer script `run_tfba_inference.py` from a terminal. The script reads `Yeast9_standard_dGr_dGbyG.csv` and generates the TFBA directionality output file `Yeast9_Directionality_TFBA.csv`.
    
    4.2. Return to this notebook to load `Yeast9_Directionality_TFBA.csv`, summarize the TFBA-derived directionality results, and identify candidate reactions with inconsistent directionality.

## Input and output files

This workflow generates and uses files at different stages.

**Initial input file**

- `yeast-GEM.xml`: Yeast-GEM SBML model used as the input GEM.

**File generated by this notebook before running ThermoInfer**

- `Yeast9_standard_dGr_dGbyG.csv`: reaction-level dGbyG prediction table. This file is generated in the first notebook section and is used as the input file for the standalone TFBA script.

**File generated by the standalone TFBA script**

- `Yeast9_Directionality_TFBA.csv`: TFBA directionality output generated by running `run_tfba_inference.py` from a terminal. After this file is generated, return to this notebook for downstream analysis.

**Final output file generated by this notebook**

- `Yeast9_candidate_inconsistent_reactions.csv`: candidate reactions whose original GEM directionality annotation may require manual review.


## 1. Load the GEM and define compartment-specific conditions in JupyterLab

This section loads the Yeast-GEM SBML model and defines the physicochemical conditions used by dGbyG to calculate transformed standard reaction Gibbs energy values.


In [1]:
import cobra
from dGbyG.api import predict_transformed_dG_prime_for_GEM

Load the Yeast-GEM SBML model.


In [2]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)

print(f"Loaded model: {gem.id}")
print(f"Number of reactions: {len(gem.reactions)}")
print(f"Number of metabolites: {len(gem.metabolites)}")

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-06
Loaded model: yeastGEM_v9__46__1__46__0
Number of reactions: 4102
Number of metabolites: 2748


Inspect the compartment identifiers in the GEM. These identifiers must match the keys used in `compartment_conditions`.


In [3]:
gem.compartments

{'ce': 'cell envelope',
 'c': 'cytoplasm',
 'e': 'extracellular',
 'm': 'mitochondrion',
 'n': 'nucleus',
 'p': 'peroxisome',
 'er': 'endoplasmic reticulum',
 'g': 'Golgi',
 'lp': 'lipid particle',
 'v': 'vacuole',
 'erm': 'endoplasmic reticulum membrane',
 'vm': 'vacuolar membrane',
 'gm': 'Golgi membrane',
 'mm': 'mitochondrial membrane'}

Define compartment-specific physicochemical conditions for dGbyG prediction.

For each compartment, `pH` defines the compartment pH, `e_potential` defines the electrical potential, `T` defines the temperature in Kelvin, `I` defines the ionic strength, and `pMg` defines the magnesium ion condition.


In [4]:
compartment_conditions = {
    "c":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "e":  {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "g":  {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "m":  {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
    "n":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "v":  {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "ce": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "p":  {"pH": 7.4,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "er": {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "lp": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "erm": {"pH": 7.2, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "vm": {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "gm": {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "mm": {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
}

Inspect the metabolite annotation types available in the GEM. dGbyG uses metabolite identifiers to obtain metabolite-level Gibbs energy predictions.


In [5]:
cid_types = sorted({
    cid_type
    for met in gem.metabolites
    for cid_type in met.annotation
})

cid_types

['bigg.metabolite',
 'biocyc',
 'chebi',
 'kegg.compound',
 'metanetx.chemical',
 'pubchem.compound',
 'reactome',
 'sbo',
 'seed.compound']

## 2. Run dGbyG and save the reaction-level thermodynamic prediction table

Run dGbyG using the loaded GEM and the compartment-specific conditions. This step generates metabolite-level and reaction-level prediction tables.


In [6]:
Met_df, Rxn_df = predict_transformed_dG_prime_for_GEM(gem,compartment_conditions=compartment_conditions)

Processing metabolites: 100%|████████████████████████████████████████████████████████| 2748/2748 [00:17<00:00, 157.86it/s]


✅ All molecules exist in the pKa database — skipping prediction.


Predicting transformed standard Gibbs free energy for metabolites: 100%|██████████████| 2748/2748 [07:10<00:00,  6.39it/s]
Predicting transformed standard Gibbs free energy for reactions: 100%|███████████████| 4102/4102 [00:23<00:00, 172.92it/s]


Preview the metabolite-level dGbyG prediction table.


In [7]:
Met_df.head()

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,pubchem.compound
s_0001,"(nan, nan)","(-424.3272705078125, 0.5277096033096313)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN
s_0002,"(nan, nan)","(-410.63547748059545, 0.5277094841003418)","(-250.62523544585773, 38.24007797241211)","(nan, nan)",NaN
s_0003,"(nan, nan)","(-424.3272705078125, 0.5277087092399597)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN
s_0004,"(nan, nan)","(-424.3272705078125, 0.527707576751709)","(-946.3900756835938, 2.3035664558410645)","(nan, nan)",NaN
s_0006,"(nan, nan)","(-1387.7450104192328, 10.174610137939453)","(-1389.32816768058, 11.83471393585205)","(nan, nan)",NaN


Preview the reaction-level dGbyG prediction table.


In [8]:
Rxn_df.head()

,dGr_prime,SD of dGr_prime
r_0001,NaN,NaN
r_0002,NaN,NaN
r_0003,15.31442,1.657002
r_0004,NaN,NaN
r_0005,NaN,NaN


Save the reaction-level dGbyG prediction table. This file is the input for the standalone ThermoInfer TFBA script.


In [9]:
Rxn_df.to_csv("./Yeast9_standard_dGr_dGbyG.csv")

## 3. Pause the notebook and run TFBA from a terminal

Stop here after `Yeast9_standard_dGr_dGbyG.csv` has been generated. Run the standalone ThermoInfer script from a terminal, not from this notebook.

```text
>python run_tfba_inference.py
```

The script reads `Yeast9_standard_dGr_dGbyG.csv` and generates `Yeast9_Directionality_TFBA.csv`. Return to the next section of this notebook only after the script finishes and the TFBA output file is generated.


## 4. Return to JupyterLab and identify candidate reactions with inconsistent directionality

This section screens for reactions whose original GEM directionality annotation may require manual review.


Load the packages needed for candidate reaction screening.


In [10]:
import numpy as np
import pandas as pd
import cobra

from ThermoInfer.utils.func import *
from dGbyG.utils.reaction_utils import build_equation

Reload the GEM and the dGbyG reaction-level prediction table.


In [11]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)

dgr_input_path = "./Yeast9_standard_dGr_dGbyG.csv"
dGr_df = pd.read_csv(dgr_input_path, index_col=0)
dGr = dGr_df[["dGr_prime", "SD of dGr_prime"]].to_numpy()

Assign multi-compartment reactions to `NaN` because these reactions are often simplified in GEMs and may differ substantially from actual transport or compartment-spanning processes. Excluding them from dGbyG-derived thermodynamic constraints helps prevent artificial infeasibility during TFBA optimization.


In [12]:
single_compartment_rxn = np.array([
    len(rxn.compartments) == 1
    for rxn in gem.reactions
])

dGr[~single_compartment_rxn, :] = np.nan


Load the TFBA directionality output.


In [13]:
tfba_output_path = "./Yeast9_Directionality_TFBA.csv"
Directionality_TFBA = pd.read_csv(tfba_output_path, index_col=0)

Directionality_TFBA.head()

,lv,uv,ldGr,udGr
rxn num,,,,
0,-1000.000000,1000.000000,-inf,inf
1,0.000000,1000.000000,-inf,inf
2,-555.557169,0.000000,-127.209806,81.193887
3,-974.630299,1000.000000,-inf,inf
4,8.122629,59.355756,-inf,inf


Build a reaction-level directionality summary table.


In [14]:
direction_df = pd.DataFrame({
    "built-in label": [rxn.reversibility for rxn in gem.reactions]
})

tfba_index = Directionality_TFBA.index.to_numpy()

direction_df.loc[tfba_index, "TFBA v direction"] = [
    direction_for_v_range(row.lv, row.uv)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[tfba_index, "TFBA dG direction"] = [
    direction_for_dGr_range(row.ldGr, row.udGr)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[:, ["dGr_prime", "SD of dGr_prime"]] = dGr

direction_df.head()


,built-in label,TFBA v direction,TFBA dG direction,dGr_prime,SD of dGr_prime
0,False,bidirection,bidirection,NaN,NaN
1,False,forward,bidirection,NaN,NaN
2,True,backward,bidirection,15.31442,1.657002
3,False,bidirection,bidirection,NaN,NaN
4,False,forward,bidirection,NaN,NaN


Select candidate reactions that are annotated as irreversible in the original GEM (typically implying forward-only flux), but whose thermodynamic assessment suggests that the annotated forward direction is unfavorable. Specifically, candidates are reactions with a TFBA thermodynamic direction assigned as backward and a TFBA flux direction that is either backward or blocked.

In [15]:
candidate_mask = (
    direction_df["built-in label"].eq(False)
    & direction_df["TFBA dG direction"].eq("backward")
    & direction_df["TFBA v direction"].isin(["backward", "blocked"])
)

Extract candidate reaction information for manual review.


In [16]:
candidate_data = []

for reaction_number in direction_df.index[candidate_mask]:
    rxn = gem.reactions[int(reaction_number)]
    stoichiometry = {met.name: coeff for met, coeff in rxn.metabolites.items()}
    compartment = sorted(list(rxn.compartments))[0] if len(rxn.compartments) > 0 else ""
    annotation = "; ".join(
        f"{key}: {value}"
        for key, value in rxn.annotation.items()
    )

    candidate_data.append([
        int(reaction_number),
        rxn.id,
        build_equation(stoichiometry, eq_sign="=>"),
        compartment,
        dGr[int(reaction_number)][0],
        dGr[int(reaction_number)][1],
        annotation,
    ])

inconsistent_reaction = pd.DataFrame(
    candidate_data,
    columns=[
        "Reaction number",
        "Reaction ID",
        "Original equation",
        "Compartment",
        "dGr_prime (kJ/mol), dGbyG",
        "SD of dGr_prime (kJ/mol), dGbyG",
        "Annotation",
    ]
).set_index("Reaction number", drop=True)

inconsistent_reaction


,Reaction ID,Original equation,Compartment,"dGr_prime (kJ/mol), dGbyG","SD of dGr_prime (kJ/mol), dGbyG",Annotation
Reaction number,,,,,,
170,r_0192,beta-D-mannosyldiacetylchitobiosyldiphosphodol...,g,-8.963946,0.937376,sbo: SBO:0000176; ec-code: 2.4.1.-; kegg.pathw...
314,r_0358,"ATP + 2.0 H+ + myo-inositol 1,3,4,5,6-pentakis...",c,48.829159,2.076522,sbo: SBO:0000176; ec-code: 2.7.4.21; kegg.path...
722,r_0953,ammonium + 2.0 H2O + 0.5 oxygen + pyridoxal =>...,c,379.561143,8.063524,sbo: SBO:0000176; ec-code: 1.4.3.5; bigg.react...
764,r_0999,iron(2+) + sirohydrochlorin => 2.0 H+ + siroheme,c,340.780562,61.200081,"sbo: SBO:0000176; ec-code: ['1.3.1.76', '4.99...."
1632,r_2256,4.0 H+ + H2O + trans-oct-2-enoyl-CoA => (R)-3-...,p,-9.781253,1.810396,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
1639,r_2263,"4.0 H+ + H2O + trans-2,cis-9-octadecadienoyl-C...",p,-22.076163,3.496447,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
4031,r_4713,2.0 ethanol + 2.0 H+ + succinate => 2.0 H2O + ...,c,37.438245,3.960187,sbo: SBO:0000176; ec-code: 3.1.-.-
4032,r_4714,ethanol + H+ + succinate => H2O + monoethyl su...,c,18.699088,1.980096,sbo: SBO:0000176; ec-code: 3.1.-.-


Save the candidate reaction table as the final output of this notebook.

In [17]:
candidate_output_path = "./Yeast9_candidate_inconsistent_reactions.csv"
inconsistent_reaction.to_csv(candidate_output_path)

print(f"Saved candidate reaction table to: {candidate_output_path}")


Saved candidate reaction table to: ./Yeast9_candidate_inconsistent_reactions.csv
